# SIPTA — Ingesta y EDA Maestro: Consolidado Distrital Multi-Sectorial
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona A (Adan Sánchez — Lead Data Engineer & Autor de EDA)**  
**Colaboración en fuentes**: Persona B (Yesid Bello — Data Scientist)  
**Objetivo**: Orquestación general de ingesta, auditoría exploratoria multi-sectorial y análisis de brechas de datos (EDA).  
**Datos de Entrada**: `data/raw/*`  
**Datos de Salida**: `data/processed/* / reports/eda/*`


## 1. Ingesta y Descubrimiento de Datos Crudos



# SIPTA Notebook: Ingestión de datos

Este notebook sirve como guía inicial para la ingesta de datos crudos en el proyecto SIPTA.

## Objetivos

- Registrar y versionar las fuentes de datos originales.
- Cargar archivos desde `data/raw` con pandas.
- Guardar copias de seguridad para reproducibilidad.

In [ ]:
import sys
from pathlib import Path
import logging

import pandas as pd

for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        break
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR


## Función de carga reutilizable

Esta función conserva la carga genérica definida para la fase ETL.

In [ ]:
def load_raw_file(filename: str, file_type: str = 'csv') -> pd.DataFrame:
    path = RAW_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo: {path}')

    if file_type.lower() == 'csv':
        return pd.read_csv(path, low_memory=False)
    if file_type.lower() == 'json':
        return pd.read_json(path)
    raise ValueError(f'Tipo no soportado: {file_type}')


In [26]:
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def download_to_raw(url: str, filename: str, subcarpeta: str = "") -> Path:
    directorio_destino = RAW_DIR / subcarpeta if subcarpeta else RAW_DIR
    directorio_destino.mkdir(parents=True, exist_ok=True)
    filepath = directorio_destino / filename
    if filepath.exists():
        logging.info(f"El archivo {filename} ya existe en {directorio_destino}. Omitiendo.")
        return filepath
    for p in RAW_DIR.rglob(filename):
        if p.exists():
            logging.info(f"El archivo {filename} localizado en {p}. Omitiendo descarga.")
            return p
    return filepath

url_poblacion = "https://datosabiertos.bogota.gov.co/dataset/85bf790d-84d1-4eda-bd6f-40af62e71d95/resource/37e58cb3-c870-4608-8c37-ce45db0eb7c1/download/osb_demografia-poblacion-localidad.csv"
archivo_poblacion = "osb_demografia-poblacion-localidad.csv"
carpeta_tematica = "DEMOGRAFIA"

filepath_poblacion = download_to_raw(url_poblacion, archivo_poblacion, subcarpeta=carpeta_tematica)

INFO: El archivo osb_demografia-poblacion-localidad.csv ya existe en C:\DataJam_DataOlinguitos_Gen\data\raw\DEMOGRAFIA. Omitiendo.


## Cargar un dataset crudo

In [27]:
def load_raw_csv(filename: str, subcarpeta: str = "", separador: str = ';') -> pd.DataFrame:
    path = RAW_DIR / subcarpeta / filename if subcarpeta else RAW_DIR / filename
    if not path.exists():
        candidates = list(RAW_DIR.rglob(filename))
        if candidates:
            path = candidates[0]
    assert path.exists(), f'No existe el archivo en la ruta: {path}'
    return pd.read_csv(path, sep=separador, low_memory=False)

df_poblacion = load_raw_csv(archivo_poblacion, subcarpeta=carpeta_tematica, separador=';')
print("Dataset cargado correctamente.")

,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
0,2005,0,Bogotá,Hombres,6,Infancia,00 a 11,67184
1,2005,0,Bogotá,Hombres,7,Infancia,00 a 11,68940
2,2005,0,Bogotá,Hombres,8,Infancia,00 a 11,70568
3,2005,0,Bogotá,Hombres,9,Infancia,00 a 11,71189
4,2005,0,Bogotá,Hombres,10,Infancia,00 a 11,70398


In [28]:
print("=== DIMENSIONES DEL DATASET ===")
print(f"Filas: {df_poblacion.shape[0]}")
print(f"Columnas: {df_poblacion.shape[1]}")

print("\n=== COLUMNAS ===")
print(df_poblacion.columns.tolist())

print("\n=== TIPOS DE DATOS ===")
print(df_poblacion.dtypes)

print("\n=== PRIMERAS 5 FILAS ===")
display(df_poblacion.head())

print("\n=== ÚLTIMAS 5 FILAS ===")
display(df_poblacion.tail())


=== DIMENSIONES DEL DATASET ===
Filas: 131502
Columnas: 8

=== COLUMNAS ===
['ANO', 'CODIGO_LOCALIDAD', 'NOMBRE_LOCALIDAD', 'SEXO', 'EDAD', 'CURSODEVIDA', 'GRUPOEDAD', 'POBLACION']

=== TIPOS DE DATOS ===
ANO                 int64
CODIGO_LOCALIDAD    int64
NOMBRE_LOCALIDAD      str
SEXO                  str
EDAD                int64
CURSODEVIDA           str
GRUPOEDAD             str
POBLACION           int64
dtype: object

=== PRIMERAS 5 FILAS ===


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
0,2005,0,Bogotá,Hombres,6,Infancia,00 a 11,67184
1,2005,0,Bogotá,Hombres,7,Infancia,00 a 11,68940
2,2005,0,Bogotá,Hombres,8,Infancia,00 a 11,70568
3,2005,0,Bogotá,Hombres,9,Infancia,00 a 11,71189
4,2005,0,Bogotá,Hombres,10,Infancia,00 a 11,70398



=== ÚLTIMAS 5 FILAS ===


,ANO,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD,SEXO,EDAD,CURSODEVIDA,GRUPOEDAD,POBLACION
131497,2035,20,Sumapaz,Mujeres,96,Vejez,60 o más,1
131498,2035,20,Sumapaz,Mujeres,97,Vejez,60 o más,0
131499,2035,20,Sumapaz,Mujeres,98,Vejez,60 o más,0
131500,2035,20,Sumapaz,Mujeres,99,Vejez,60 o más,0
131501,2035,20,Sumapaz,Mujeres,100,Vejez,60 o más,0


In [29]:
print("=== CÓDIGOS DE LOCALIDAD ===")
print(sorted(df_poblacion["CODIGO_LOCALIDAD"].dropna().unique()))

print("\n=== NOMBRES DE LOCALIDAD ===")
print(sorted(df_poblacion["NOMBRE_LOCALIDAD"].dropna().unique()))

print("\n=== CANTIDAD DE CÓDIGOS ÚNICOS ===")
print(df_poblacion["CODIGO_LOCALIDAD"].nunique())

print("\n=== CANTIDAD DE NOMBRES ÚNICOS ===")
print(df_poblacion["NOMBRE_LOCALIDAD"].nunique())

print("\n=== RELACIÓN CÓDIGO - LOCALIDAD ===")
display(
    df_poblacion[
        ["CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD"]
    ]
    .drop_duplicates()
    .sort_values("CODIGO_LOCALIDAD")
)


=== CÓDIGOS DE LOCALIDAD ===
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]

=== NOMBRES DE LOCALIDAD ===
['Antonio Nariño', 'Barrios Unidos', 'Bogotá', 'Bosa', 'Chapinero', 'Ciudad Bolívar', 'Engativá', 'Fontibón', 'Kennedy', 'La Candelaria', 'Los Mártires', 'Puente Aranda', 'Rafael Uribe Uribe', 'San Cristóbal', 'Santa Fe', 'Suba', 'Sumapaz', 'Teusaquillo', 'Tunjuelito', 'Usaquén', 'Usme']

=== CANTIDAD DE CÓDIGOS ÚNICOS ===
21

=== CANTIDAD DE NOMBRES ÚNICOS ===
21

=== RELACIÓN CÓDIGO - LOCALIDAD ===


,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD
0,0,Bogotá
6,1,Usaquén
12,2,Chapinero
18,3,Santa Fe
24,4,San Cristóbal
30,5,Usme
36,6,Tunjuelito
42,7,Bosa
48,8,Kennedy
54,9,Fontibón


In [31]:
print("=== AÑOS DISPONIBLES ===")

print(sorted(df_poblacion["ANO"].dropna().unique()))

print("\nAño mínimo:")
print(df_poblacion["ANO"].min())

print("\nAño máximo:")
print(df_poblacion["ANO"].max())

print("\nCantidad de años distintos:")
print(df_poblacion["ANO"].nunique())


=== AÑOS DISPONIBLES ===
[np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

Año mínimo:
2005

Año máximo:
2035

Cantidad de años distintos:
31


In [32]:
print("=== VALORES NULOS POR COLUMNA ===")

nulos = df_poblacion.isna().sum()

porcentaje_nulos = (
    df_poblacion.isna().mean() * 100
).round(2)

validacion_nulos = (
    pd.DataFrame({
        "nulos": nulos,
        "porcentaje": porcentaje_nulos
    })
    .sort_values("porcentaje", ascending=False)
)

display(validacion_nulos)


=== VALORES NULOS POR COLUMNA ===


,nulos,porcentaje
ANO,0,0.0
CODIGO_LOCALIDAD,0,0.0
NOMBRE_LOCALIDAD,0,0.0
SEXO,0,0.0
EDAD,0,0.0
CURSODEVIDA,0,0.0
GRUPOEDAD,0,0.0
POBLACION,0,0.0


In [33]:
print("=== DUPLICADOS EXACTOS ===")

duplicados_exactos = df_poblacion.duplicated().sum()

print(f"Filas duplicadas exactas: {duplicados_exactos}")

porcentaje_duplicados = (
    duplicados_exactos / len(df_poblacion) * 100
)

print(
    f"Porcentaje de duplicados: "
    f"{porcentaje_duplicados:.4f}%"
)


=== DUPLICADOS EXACTOS ===
Filas duplicadas exactas: 0
Porcentaje de duplicados: 0.0000%


## Guardar versión raw con metadatos

In [10]:
def save_versioned_raw(df: pd.DataFrame, filename: str, suffix: str = 'v1') -> Path:
    output = RAW_DIR / f'{filename.stem}_{suffix}{filename.suffix}'
    df.to_csv(output, index=False)
    return output

# Ejemplo de uso:
# save_versioned_raw(df, Path('dataset.csv'), suffix='20260731')



## Guardar una versión procesada

In [ ]:
def save_processed_copy(df: pd.DataFrame, output_name: str) -> Path:
    destination = PROCESSED_DIR / output_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(destination, index=False)
    return destination


## Notas de ingesta — Demografía

- La fuente demográfica fue descargada y almacenada en
  `data/raw/DEMOGRAFIA/osb_demografia-poblacion-localidad.csv`.

- El archivo crudo debe conservarse sin modificaciones manuales para mantener
  la trazabilidad y reproducibilidad del análisis.

- El CSV utiliza `;` como separador y se carga mediante codificación
  `utf-8-sig`.

- Durante las pruebas iniciales se detectó un Byte Order Mark (BOM) asociado
  al encabezado `ANO`; la lectura con `utf-8-sig` permite interpretar
  correctamente el nombre de la columna sin modificar el archivo original.

- La carga completa produce 131.502 registros y 8 variables.

- Los metadatos de la fuente se almacenan por separado en `data/external`
  cuando corresponda.

- Este notebook se utiliza para probar y documentar el proceso de ingesta.
  Una vez estabilizada la lógica reutilizable, podrá trasladarse a
  `src/ingestion/`.

- Las transformaciones, correcciones o reglas de calidad no deben aplicarse
  sobre `data/raw`; deben realizarse en etapas posteriores del pipeline.

## 2. Análisis Exploratorio Multi-Sectorial



# 00 · EDA Maestro — inventario, calidad, cobertura e indicadores

Síntesis transversal del EDA de SIPTA: consolida los resultados de los notebooks 01 a 07
y genera los reportes canónicos en `reports/eda/`:

| Reporte | Contenido |
|---|---|
| `perfil_datos.csv` | Perfil de calidad por archivo/capa (nulos, columnas, problemas) |
| `perfil_variables.csv` | Perfil estadístico de cada variable explorada (consolida notebooks 01-07) |
| `matriz_cobertura_localidad.csv` | Cobertura territorial por localidad x fuente |
| `resumen_indicadores_eda.csv` | Catálogo de indicadores con estado de construcción |
| `conclusiones_eda.md` | Conclusiones con métricas |
| `tiempos_ejecucion.csv` | Tiempos por sección de todos los notebooks |
| `inventario_fuentes.csv` | Inventario físico de `data/raw` |

> **Orden de ejecución**: primero los sectores (01 → 07) y por último este maestro,
> porque consolida los perfiles por variable de los demás.

## 0. Configuración

In [1]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")


ROOT: C:\Users\ADAN\DataJam_DataOlinguitos_Gen | SMOKE: False


## 1. Inventario físico vs catálogo

In [2]:
t0("inventario")
catalog = eda.load_catalog()
approved = eda.load_approved()
print("catálogo:", len(catalog), "fuentes | aprobadas:", len(approved))

filas_inv = []
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file() and p.suffix not in (".md", ".gitkeep"):
        rel = p.relative_to(RAW_DIR)
        filas_inv.append({
            "archivo": str(rel),
            "sector": rel.parts[0],
            "formato": p.suffix.lstrip(".").upper() or "SIN EXT",
            "tam_mb": round(p.stat().st_size / 1e6, 2),
        })
inventario = pd.DataFrame(filas_inv)
inventario.to_csv(REPORTS / "inventario_fuentes.csv", index=False)
display(inventario)
print(f"Archivos físicos: {len(inventario)} | sectores: {inventario['sector'].nunique()}")
display(inventario.groupby(["sector", "formato"]).size().unstack(fill_value=0))
t1("inventario")


catálogo: 23 fuentes | aprobadas: 23


,archivo,sector,formato,tam_mb
0,.gitkeep,.gitkeep,SIN EXT,0.00
1,AMBIENTE\estacion_calidad_aire.geojson,AMBIENTE,GEOJSON,0.01
2,AMBIENTE\situacion_ambiental_conflictiva.csv,AMBIENTE,CSV,0.29
3,AMBIENTE\situacion_ambiental_conflictiva.geojson,AMBIENTE,GEOJSON,0.63
4,DEMOGRAFIA\osb_demografia-poblacion-localidad.csv,DEMOGRAFIA,CSV,6.82
...,...,...,...,...
96,MOVILIDAD\zonas_zat.geojson,MOVILIDAD,GEOJSON,27.34
97,SALUD\ips_sds.gpkg,SALUD,GPKG,1.38
98,SALUD\osb_ofertasrv-ips-urgencias.csv,SALUD,CSV,0.02
99,SALUD\osb_tiporazoncamas.csv,SALUD,CSV,0.00


Archivos físicos: 101 | sectores: 10


formato,CSV,GEOJSON,GPKG,SIN EXT,TXT,XLSX,ZIP
sector,,,,,,,
.gitkeep,0,0,0,1,0,0,0
AMBIENTE,1,2,0,0,0,0,0
DEMOGRAFIA,1,0,0,0,0,0,0
DEMOGRAFIA_POBLACION,2,0,0,0,0,0,0
EDUCACION,0,1,2,0,0,0,0
FINANZAS_INVERSION_PUBLICA,0,1,1,0,6,1,0
INFRAESTRUCTURA_ESPACIO_PUBLICO,2,0,2,0,0,0,0
MOVILIDAD,2,6,3,0,0,62,1
SALUD,2,0,1,0,0,0,0


## 2. Perfil global de calidad por archivo

In [3]:
t0("perfil_global")
from src.eda.quality import profile_file
filas_q = []
candidate_files = [p for p in sorted(RAW_DIR.rglob("*")) if p.is_file() and p.suffix not in (".md", ".gitkeep")]
if SMOKE:
    candidate_files = candidate_files[:10]

for p in candidate_files:
    rel = str(p.relative_to(RAW_DIR))
    try:
        for e in profile_file(p, ROOT / "data", smoke=SMOKE):
            e["archivo"] = rel
            filas_q.append(e)
    except Exception as ex:
        filas_q.append({"archivo": rel, "error": str(ex)[:200]})
perfil_datos = pd.DataFrame(filas_q)
perfil_datos.to_csv(REPORTS / "perfil_datos.csv", index=False)
print(f"Entradas perfiladas: {len(perfil_datos)}")
cols_mostrar = [c for c in ["archivo", "capa_hoja", "filas_totales", "columnas", "pct_nulos_total", "problemas_detectados"] if c in perfil_datos.columns]
display(perfil_datos[cols_mostrar])
if "problemas_detectados" in perfil_datos.columns:
    con_problemas = perfil_datos[perfil_datos["problemas_detectados"].astype(str).str.len() > 2]
    print("Entradas con problemas:", len(con_problemas))
t1("perfil_global")

Entradas perfiladas: 276


,archivo,capa_hoja,filas_totales,columnas,pct_nulos_total,problemas_detectados
0,.gitkeep,NaN,NaN,NaN,NaN,formato_no_perfilado
1,AMBIENTE\estacion_calidad_aire.geojson,NaN,19.0,12.0,21.49,sin_columna_territorial
2,AMBIENTE\situacion_ambiental_conflictiva.csv,NaN,NaN,14.0,9.01,"duplicados, encoding_legacy"
3,AMBIENTE\situacion_ambiental_conflictiva.geojson,NaN,1373.0,10.0,0.01,duplicados
4,DEMOGRAFIA\osb_demografia-poblacion-localidad.csv,NaN,NaN,8.0,0.00,sin_alertas
...,...,...,...,...,...,...
271,MOVILIDAD\zonas_zat.geojson,NaN,2356.0,6.0,0.00,"sin_columna_territorial, bbox_fuera_bogota"
272,SALUD\ips_sds.gpkg,ips,2900.0,27.0,0.00,sin_alertas
273,SALUD\osb_ofertasrv-ips-urgencias.csv,NaN,NaN,11.0,0.11,"sin_columna_territorial, encoding_legacy"
274,SALUD\osb_tiporazoncamas.csv,NaN,NaN,4.0,18.10,"duplicados, sin_columna_territorial, encoding_..."


Entradas con problemas: 276


## 3. Matriz de cobertura territorial

Cruce de cada fuente con dimensión territorial contra las 20 localidades (+ Bogotá). Los puntos sin columna de localidad (IPS, estaciones) se asignan por cruce espacial con la capa Loca del MR.

In [4]:
t0("matriz_cobertura")
from src.eda.profiling import localidad_de_codigo, LOCALIDADES_20
from src.eda.spatial import load_loca, count_points_by_locality

def conteo_serie(df, col, es_codigo=False):
    mapper = localidad_de_codigo if es_codigo else eda.standardize_locality
    s = df.groupby(col).size()
    s.index = s.index.map(mapper)
    return s[s.index.notna()]

loca = load_loca(MR_PATH)
matriz = pd.DataFrame(index=sorted(LOCALIDADES_20))

demo = pd.read_csv(str(RAW_DIR / "DEMOGRAFIA_POBLACION" / "osb_demografia-poblacion-localidad.csv"), sep=";", encoding="utf-8")
demo = demo[demo["ANO"] == demo["ANO"].max()]
matriz["poblacion_demografia"] = conteo_serie(demo, "NOMBRE_LOCALIDAD")

parq = eda.read_csv_robust(str(RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "5.-parques-idrd.csv"))[0]
tcols = eda.detect_territorial_columns(parq)
if tcols:
    matriz["parques_idrd"] = conteo_serie(parq, tcols[0])

par = gpd.read_file(str(RAW_DIR / "MOVILIDAD" / "paraderos_zonales_sitp.gpkg"), layer="Paraderos")
tcols = eda.detect_territorial_columns(par)
if tcols:
    matriz["paraderos_sitp"] = conteo_serie(par, tcols[0])

col = gpd.read_file(str(RAW_DIR / "EDUCACION" / "colegios122025.gpkg"), layer="colegios_122025")
matriz["sedes_educativas"] = conteo_serie(col, "COD_LOCA", es_codigo=True)

mat = gpd.read_file(str(RAW_DIR / "EDUCACION" / "matricula_total_colegios_oficiales.gpkg"), layer="matriculatotal_042025")
matriz["matricula_oficial"] = conteo_serie(mat, "COD_LOCA", es_codigo=True)

fin = gpd.read_file(str(RAW_DIR / "FINANZAS_INVERSION_PUBLICA" / "inversion_educacion_por_localidad_12_2025.gpkg"), layer="sed_2026__territorializacioninversion2025")
matriz["inversion_educacion_2025"] = conteo_serie(fin, "COD_LOCA", es_codigo=True)

ips = gpd.read_file(str(RAW_DIR / "SALUD" / "ips_sds.gpkg"), layer="ips")
matriz["ips_salud"] = count_points_by_locality(ips, loca).set_index("localidad")["n"]

tr = gpd.read_file(str(RAW_DIR / "MOVILIDAD" / "estaciones_troncales.geojson"))
matriz["estaciones_troncales"] = count_points_by_locality(tr, loca).set_index("localidad")["n"]

l1 = gpd.read_file(str(RAW_DIR / "MOVILIDAD" / "estaciones_linea1.geojson"))
matriz["estaciones_linea1"] = count_points_by_locality(l1, loca).set_index("localidad")["n"]

l2 = gpd.read_file(str(RAW_DIR / "MOVILIDAD" / "estaciones_linea2.gpkg"), layer="estacionessegundalinea")
matriz["estaciones_linea2"] = count_points_by_locality(l2, loca).set_index("localidad")["n"]

matriz = matriz.fillna(0).astype(int)
matriz["total_fuentes_con_dato"] = (matriz > 0).sum(axis=1)
matriz.to_csv(REPORTS / "matriz_cobertura_localidad.csv", encoding="utf-8-sig")
display(matriz)
print("Cobertura por fuente (% de 20 localidades):")
print((matriz.drop(columns="total_fuentes_con_dato") > 0).mean(axis=0).sort_values(ascending=False).map(lambda x: f"{x:.0%}").to_string())
t1("matriz_cobertura")


,poblacion_demografia,parques_idrd,paraderos_sitp,sedes_educativas,matricula_oficial,inversion_educacion_2025,ips_salud,estaciones_troncales,estaciones_linea1,estaciones_linea2,total_fuentes_con_dato
Antonio Nariño,202,3,92,42,12,1,69,5,0,0,8
Barrios Unidos,202,10,234,63,23,1,210,13,0,2,9
Bosa,202,11,485,147,63,1,46,5,0,0,8
Chapinero,202,3,333,26,7,1,511,9,0,0,8
Ciudad Bolívar,202,14,593,149,81,1,50,2,0,0,8
Engativá,202,13,784,258,69,1,172,12,0,5,9
Fontibón,202,5,388,97,21,1,137,2,0,0,8
Kennedy,202,13,926,261,75,1,193,10,0,0,8
La Candelaria,202,2,37,19,3,1,10,1,0,0,8
Los Mártires,202,4,167,35,14,1,34,9,0,0,8


Cobertura por fuente (% de 20 localidades):
poblacion_demografia        100%
sedes_educativas            100%
inversion_educacion_2025    100%
matricula_oficial           100%
ips_salud                   100%
parques_idrd                 95%
paraderos_sitp               95%
estaciones_troncales         95%
estaciones_linea2            15%
estaciones_linea1             0%


## 4. Catálogo de indicadores y estado

In [5]:
t0("indicadores")
sts = eda.indicator_status()
sts.to_csv(REPORTS / "resumen_indicadores_eda.csv", index=False)
display(sts[["indicador", "dimension", "estado", "que_falta"]])
print(sts.groupby("estado").size().to_string())
t1("indicadores")


,indicador,dimension,estado,que_falta
0,Población total por localidad,Demografía,construible_ahora,Nada: fuente por localidad 2005-2035.
1,Población 0-17 años por localidad,Demografía,construible_ahora,Nada: EDAD viene desagregada.
2,Población 60+ por localidad,Demografía,construible_ahora,Nada.
3,Índice de dependencia y envejecimiento por loc...,Demografía,construible_ahora,Nada.
4,IPS por localidad (acceso a salud),Salud,construible_con_cruce_espacial,IPS no trae localidad explícita; requiere sjoi...
5,Camas hospitalarias por 10.000 habitantes,Salud,faltante_territorial,Fuente distrital sin desagregación por localid...
6,Sedes educativas por localidad,Educación,construible_ahora,Nada: COD_LOCA presente.
7,Matrícula oficial por localidad y por 1.000 ni...,Educación,construible_ahora,Nada: COD_LOCA y TMATRIC_GE presentes.
8,Paraderos SITP por 10.000 habitantes,Movilidad,construible_ahora,Nada: campo localidad_ con 20 localidades.
9,Estaciones troncales por localidad,Movilidad,construible_con_cruce_espacial,Estaciones sin localidad explícita; requiere s...


estado
construible_ahora                 11
construible_con_cruce_espacial     2
construible_parcial                2
faltante                           5
faltante_territorial               1


## 5. Conclusiones

In [6]:
t0("conclusiones")
lines = []
lines.append("# Conclusiones del EDA — SIPTA Bogotá")
lines.append("")
lines.append(f"Generado: {pd.Timestamp.now():%Y-%m-%d %H:%M} | modo smoke: {SMOKE}")
lines.append("")
lines.append("## 1. Inventario y calidad")
lines.append("")
lines.append(f"- **{len(inventario)}** archivos físicos en `data/raw` en {inventario['sector'].nunique()} sectores.")
if "pct_nulos_total" in perfil_datos.columns:
    pr = perfil_datos["pct_nulos_total"].dropna()
    lines.append(f"- % de nulos medio por fuente: **{pr.mean():.1f}%** (mediana {pr.median():.1f}%).")
lines.append("")
lines.append("## 2. Cobertura territorial")
lines.append("")
lines.append(f"- Matriz de cobertura: {matriz.shape[0]} localidades x {matriz.shape[1] - 1} fuentes; en promedio cada fuente cubre **{(matriz > 0).mean(axis=0).mean():.0%}** de las localidades.")
lines.append(f"- Fuentes con dato en las 20 localidades: {(matriz.drop(columns='total_fuentes_con_dato') > 0).all(axis=0).sum()}.")
lines.append("")
lines.append("## 3. Indicadores")
lines.append("")
for est, n in sts.groupby("estado").size().items():
    lines.append(f"- **{est}**: {n}")
lines.append("")
lines.append("## 4. Hallazgos clave")
lines.append("")
hallazgos = [
    "La población por localidad (OSB, serie anual) permite calcular denominadores per cápita para todos los sectores: es la fuente más transversal del proyecto.",
    "Los sectores AMBIENTE, PARTICIPACIÓN CIUDADANA, SEGURIDAD y SERVICIOS PÚBLICOS no tienen datos físicos: sus indicadores (AMB-01, PAR-01, SEG-01, SER-01) quedan como faltantes.",
    "El conteo territorial de IPS y estaciones se resolvió con cruce espacial contra la capa Loca del mapa de referencia (antes 0/20 localidades).",
    "Los XLSX de validaciones de TransMilenio tienen filas de título arriba del encabezado: el lector robusto las omite y el % de nulos reportado es real.",
    "El mapa de referencia (MR, 41 capas) se usa como base cartográfica para cruces, no como fuente de indicadores.",
    "Los 62 archivos mensuales de validaciones TM permiten construir series diarias y mensuales de demanda por modo.",
]
lines.extend(f"- {h}" for h in hallazgos)
lines.append("")
lines.append("## 5. Siguientes pasos")
lines.append("")
lines.append("- Priorizar la consecución de fuentes de los sectores vacíos (ver `07_eda_gaps.ipynb`).")
lines.append("- Construir los indicadores marcados como `construible_ahora` (ver `resumen_indicadores_eda.csv`).")
lines.append("- Validar las unidades monetarias de inversión (R_ASIGNADOS) con la SED antes de publicar.")
texto = "\n".join(lines)
(REPORTS / "conclusiones_eda.md").write_text(texto, encoding="utf-8")
print(texto)
t1("conclusiones")


# Conclusiones del EDA — SIPTA Bogotá

Generado: 2026-08-18 10:05 | modo smoke: False

## 1. Inventario y calidad

- **101** archivos físicos en `data/raw` en 10 sectores.
- % de nulos medio por fuente: **12.4%** (mediana 0.0%).

## 2. Cobertura territorial

- Matriz de cobertura: 20 localidades x 10 fuentes; en promedio cada fuente cubre **82%** de las localidades.
- Fuentes con dato en las 20 localidades: 5.

## 3. Indicadores

- **construible_ahora**: 11
- **construible_con_cruce_espacial**: 2
- **construible_parcial**: 2
- **faltante**: 5
- **faltante_territorial**: 1

## 4. Hallazgos clave

- La población por localidad (OSB, serie anual) permite calcular denominadores per cápita para todos los sectores: es la fuente más transversal del proyecto.
- Los sectores AMBIENTE, PARTICIPACIÓN CIUDADANA, SEGURIDAD y SERVICIOS PÚBLICOS no tienen datos físicos: sus indicadores (AMB-01, PAR-01, SEG-01, SER-01) quedan como faltantes.
- El conteo territorial de IPS y estaciones se resolvió con c

## 6. Exportación final: perfil de variables y tiempos

In [7]:
guardar_tiempos("00_eda_maestro.csv")

csvs = sorted(PERFILES.glob("*.csv"))
valid_dfs = []
for c in csvs:
    if "__territorial" not in c.name and c.stat().st_size > 0:
        try:
            df_c = pd.read_csv(c)
            if not df_c.empty:
                valid_dfs.append(df_c)
        except Exception:
            pass

if valid_dfs:
    perfil_variables = pd.concat(valid_dfs, ignore_index=True)
    perfil_variables.to_csv(REPORTS / "perfil_variables.csv", index=False)
    print("Variables perfiladas:", len(perfil_variables))
else:
    print("Sin perfiles por variable: ejecute primero los notebooks 01-07.")

times_files = sorted(TIEMPOS.glob("*.csv"))
valid_times = []
for t in times_files:
    if t.stat().st_size > 0:
        try:
            df_t = pd.read_csv(t)
            if not df_t.empty:
                valid_times.append(df_t)
        except Exception:
            pass

if valid_times:
    times = pd.concat(valid_times, ignore_index=True)
    times.to_csv(REPORTS / "tiempos_ejecucion.csv", index=False)
    display(times)
    print("Total segundos:", f"{times['segundos'].sum():,.0f}")
else:
    print("Sin tiempos registrados.")

tiempos guardados: 00_eda_maestro.csv (5 secciones)


EmptyDataError: No columns to parse from file

## Resumen ejecutivo

- **Datos**: la mayoría de los sectores con datos físicos tienen cobertura territorial real
  (demografía, educación, finanzas, parques, paraderos) o cruce espacial resuelto (IPS, estaciones).
- **Calidad**: los lectores robustos de `src/eda` eliminan los problemas de lectura (CSV con `;`,
  XLSX con filas de título) que antes inflaban los nulos al 300%.
- **Brechas**: 4 sectores sin datos y 21 indicadores del catálogo con estado definido:
  los `construible_ahora` listos para la fase de indicadores.
- **Salidas**: todos los reportes en `reports/eda/`; conclusiones en `conclusiones_eda.md`.

## 3. Análisis de Brechas y Factibilidad Territorial



# 07 · EDA Brechas de datos (gaps)

Cuatro sectores del catálogo **no tienen datos físicos** en `data/raw/`:
- `AMBIENTE`, `PARTICIPACION_CIUDADANA` y `SEGURIDAD` — solo contienen `README.md`.
- `SERVICIOS_PUBLICOS` — vacío.

Este notebook documenta qué promete el catálogo, qué indicadores quedan sin construir y
qué se necesita para cerrar la brecha.

## 0. Configuración

In [ ]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")


## 1. Sectores sin datos

In [ ]:
sin_datos = ["AMBIENTE", "PARTICIPACION_CIUDADANA", "SEGURIDAD", "SERVICIOS_PUBLICOS"]
filas = []
for s in sin_datos:
    d = RAW_DIR / s
    archivos = [str(p.relative_to(d)) for p in d.rglob("*") if p.is_file() and p.suffix not in (".md", ".gitkeep")]
    readme = d / "README.md"
    desc = ""
    if readme.exists():
        txt = readme.read_text(encoding="utf-8", errors="ignore").strip()
        desc = txt.splitlines()[0] if txt else ""
    filas.append({"sector": s, "archivos": archivos, "descripcion_readme": desc})
gaps = pd.DataFrame(filas)
display(gaps)
gaps.to_csv(REPORTS / "gaps_sectores_sin_datos.csv", index=False)


## 2. Indicadores que quedan sin construir

In [ ]:
sts = eda.indicator_status()
falt = sts[sts["estado"].astype(str).str.contains("faltante", case=False, na=False)]
display(falt[["indicador", "dimension", "estado", "que_falta"]])
falt.to_csv(REPORTS / "indicadores_faltantes.csv", index=False)
print(sts.groupby("estado").size().to_string())


## 3. Recomendaciones

1. **Conseguir las fuentes prometidas en los README** de AMBIENTE, PARTICIPACIÓN CIUDADANA y
   SEGURIDAD: sin datos físicos no hay indicadores (AMB-01, PAR-01, SEG-01).
2. **SERVICIOS_PUBLICOS** no tiene ni README: definir primero qué se va a medir (cobertura de
   acueducto, alcantarillado, energía, aseo) y de dónde saldrá (SDA, EAAB, Enel, Uaesp).
3. **Indicadores con cruce espacial pendiente** (p. ej. delitos por localidad, calidad de aire
   por localidad): requieren fuentes por localidad o geometrías georreferenciadas.
4. Priorizar el cierre de los indicadores `faltante_territorial` porque el MR ya permite
   georreferenciar cualquier punto a su localidad.

## 4. Tiempos

In [ ]:
guardar_tiempos("07_eda_gaps.csv")
